# 07 · Tokenization and Embeddings

In plain English, a language model can't actually *read*. It only understands numbers. So before any text reaches a model, we have to translate it into numbers — and after the model answers, we translate those numbers back into text. This notebook walks the whole pipeline: **raw text → tokens → token IDs → embeddings (vectors)**. We'll do it hands-on with real Hugging Face tokenizers, using tiny, free models that run fine on a laptop CPU. By the end you'll understand exactly what happens when you "tokenize your dataset" — which is the very first step of every fine-tuning project.

## What you'll learn

- **Why** models need numbers instead of raw text.
- What a **token** is, and the difference between **word-level** and **subword** tokenizers (BPE / WordPiece) — and why subword tokenizers gracefully handle rare or never-before-seen words.
- How to use Hugging Face `AutoTokenizer` with two small models:
  - `bert-base-uncased` (a BERT-style model, WordPiece).
  - `distilgpt2` (a GPT-style model, byte-level BPE).
- The three workhorse methods: `tokenizer.tokenize(text)`, `tokenizer(text)` (which returns `input_ids` + `attention_mask`), and `tokenizer.decode(...)`.
- **Special tokens** like `[CLS]`, `[SEP]`, the padding token, and the end-of-sequence (`eos`) token — what they're for.
- **Padding**, **truncation**, `max_length`, and the **attention mask** (which tokens are real vs. just filler).
- What an **embedding** is: a learned vector per token id (a lookup table), demonstrated with `torch.nn.Embedding`, plus a quick **cosine similarity** to build the intuition "similar meaning ≈ closer vectors."

## Why this matters for fine-tuning

Tokenization isn't a side topic — **preparing data for fine-tuning *is* tokenizing it**. Three things you'll meet over and over:

- **Your dataset must be tokenized** before training. Every example becomes `input_ids` and an `attention_mask`. If you've ever seen a `dataset.map(tokenize_function)` step, that's this notebook in action.
- **`max_length` controls memory and speed.** Longer sequences mean more compute and more GPU/CPU memory per example. Choosing a sensible `max_length` (and truncating) is one of the easiest ways to make fine-tuning fit on your hardware.
- **The tokenizer must match the model.** A very common (and confusing) bug is loading `distilgpt2`'s tokenizer but a BERT model, or vice versa. The token IDs won't line up with what the model learned, and results will be garbage. Rule of thumb: **load the tokenizer from the same checkpoint as the model.**

Embeddings matter because they're the model's *first layer* — and during fine-tuning, those embedding vectors can keep learning, nudging the meanings of tokens toward your task.

## Setup

Run the cell below once. The `%pip install` line is **commented out** — uncomment it if you're on Google Colab or a fresh environment.

The first time you load a tokenizer or model, Hugging Face **downloads it once** from the Hugging Face Hub and caches it on disk (it's **free**, no account needed for these public models). After that, loading is instant and works offline. The models here are small and CPU-friendly.

In [ ]:
# Uncomment the next line on Colab or a fresh environment:
# %pip install transformers torch

import torch                              # PyTorch: the tensor (number-grid) library models run on
from transformers import AutoTokenizer    # auto-loads the correct tokenizer for any model checkpoint

print("transformers + torch imported OK")  # -> transformers + torch imported OK

## 1. Why models can't read raw text

A neural network is, under the hood, a big pile of **multiplications and additions on numbers**. The letters `"c"`, `"a"`, `"t"` mean nothing to it. So step one of *every* NLP system is to convert text into integers, then into vectors of numbers.

The pipeline looks like this:

```
"I love cats"  ->  ["i", "love", "cats"]   ->  [1045, 2293, 8870]  ->  [[0.12, -0.4, ...], [...], [...]]
   raw text          tokens                     token IDs (ints)        embeddings (vectors)
```

Each arrow is a stage we'll cover. The **tokenizer** handles the first two arrows (text → tokens → IDs). The **model's embedding layer** handles the last arrow (IDs → vectors).

In [ ]:
# A toy "tokenizer" to build intuition (real ones are smarter — see below).
sentence = "I love cats"

# Step 1: split text into pieces ("tokens"). Here we just split on spaces.
toy_tokens = sentence.lower().split()
print("tokens:", toy_tokens)        # -> tokens: ['i', 'love', 'cats']

# Step 2: map each token to an integer id using a tiny made-up vocabulary.
toy_vocab = {"i": 1045, "love": 2293, "cats": 8870}
toy_ids = [toy_vocab[t] for t in toy_tokens]
print("token ids:", toy_ids)        # -> token ids: [1045, 2293, 8870]

**What this does:** It shows the *idea* of tokenization in pure Python — split text into tokens, then look each token up in a vocabulary to get an integer id. A real tokenizer does exactly this, but with a much larger, learned vocabulary and smarter splitting rules so it never gets stuck on a word it hasn't seen.

### ✏️ Exercise

Add the word `"dogs"` to `toy_vocab` (give it any integer id), then tokenize the sentence `"I love dogs"` with the toy tokenizer. What happens if you try to tokenize `"I adore cats"` *without* adding `"adore"` to the vocabulary? (Try it — the error you get is exactly the "unknown word" problem real tokenizers are designed to avoid.)

## 2. Word-level vs. subword tokenizers

Our toy tokenizer has a fatal flaw: if a word isn't in the vocabulary, it breaks. Real text is full of typos, names, slang, and rare words. A **word-level** tokenizer (one id per whole word) would need a gigantic vocabulary and *still* fail on anything new — those become a single "unknown" token (`[UNK]`), and the model loses all information about them.

**Subword tokenizers** solve this. Instead of whole words, they learn a vocabulary of common *word-pieces*. Two popular algorithms:

- **BPE (Byte-Pair Encoding)** — used by GPT-style models (`distilgpt2`). It starts from characters and repeatedly merges the most frequent pairs into bigger pieces.
- **WordPiece** — used by BERT-style models (`bert-base-uncased`). Similar idea, slightly different merge rule.

The payoff: a rare word like `"tokenization"` might split into `token` + `##ization`, and a totally novel word can always be built from smaller pieces (down to single characters). **Nothing is ever truly "unknown."** That's why every modern LLM uses subword tokenization.

## 3. Hands-on: loading a real tokenizer (BERT / WordPiece)

`AutoTokenizer.from_pretrained("...")` downloads and builds the exact tokenizer that a given model was trained with. We'll start with `bert-base-uncased`. "Uncased" means it lowercases everything (so `"Cat"` and `"cat"` are the same).

In [ ]:
# Download (once) and load BERT's tokenizer. ~few hundred KB, very fast.
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")

print("vocab size:", bert_tok.vocab_size)   # -> vocab size: 30522 (number of known tokens)
print("type:", type(bert_tok).__name__)      # -> something like BertTokenizerFast

**What this does:** `from_pretrained` fetches the tokenizer files (the vocabulary and the merge/split rules) for `bert-base-uncased` and returns a ready-to-use tokenizer object. The **vocab size** (~30,522) is how many distinct token pieces this tokenizer knows. Every piece of text you feed it will be expressed using only those ~30k tokens.

In [ ]:
# .tokenize() shows the human-readable TOKENS (before turning them into ids).
text = "I love tokenization!"
tokens = bert_tok.tokenize(text)
print(tokens)
# -> ['i', 'love', 'token', '##ization', '!']
# Notice: "tokenization" was SPLIT into 'token' + '##ization'.
# The '##' prefix means "this piece attaches to the previous token (no space before it)."

**What this does:** `tokenize()` returns the actual subword tokens as strings, *without* converting them to numbers yet — perfect for seeing what the tokenizer is doing. The rare word `"tokenization"` is broken into the common piece `token` plus the suffix `##ization`. The `##` is WordPiece's way of saying "glue me onto the previous token." This is subword tokenization in action: a single word became **two** tokens.

In [ ]:
# Try a made-up / rare word to see subword splitting really shine.
print(bert_tok.tokenize("antidisestablishmentarianism"))
# -> ['anti', '##dis', '##est', '##ab', '##lish', '##ment', '##arian', '##ism']  (or similar)

print(bert_tok.tokenize("Claude2024xyz"))
# -> ['claude', '##2024', '##xy', '##z']  (or similar) -- still no [UNK]!
# Even a long/fake word becomes known pieces instead of being dropped as unknown.
# That's why your domain terms (product names, code identifiers) are always representable.

### ✏️ Exercise

Tokenize three words of your own choosing with `bert_tok.tokenize(...)`: one common word, one rare/technical word, and one made-up word. Count how many tokens each produces. Which ones got split with `##`? This is a great way to *feel* how subword tokenization works.

## 4. From tokens to model inputs: `input_ids` and `attention_mask`

`tokenize()` is for *looking*. To actually feed a model, you **call the tokenizer like a function**: `bert_tok(text)`. This does everything at once and returns a dictionary with:

- `input_ids` — the integer id for each token (this is what the model consumes).
- `attention_mask` — a list of 1s and 0s telling the model which tokens are **real** (`1`) vs. **padding** (`0`). With a single sentence and no padding, it's all 1s.

It also automatically adds **special tokens** (more on those next).

In [ ]:
encoded = bert_tok("I love tokenization!")
print(encoded)
# Expected (something like):
# {'input_ids': [101, 1045, 2293, 19204, 3989, 999, 102],
#  'token_type_ids': [0, 0, 0, 0, 0, 0, 0],
#  'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

print("input_ids:     ", encoded["input_ids"])
print("attention_mask:", encoded["attention_mask"])

**What this does:** Calling `bert_tok(text)` returns the ready-for-model encoding. Notice `input_ids` starts with **101** and ends with **102** — those are BERT's special `[CLS]` and `[SEP]` tokens that the tokenizer added for you. The `attention_mask` is all `1`s because every token is real (we didn't pad anything). `token_type_ids` is a BERT-specific extra used to tell apart two sentences; you can ignore it for now.

In [ ]:
# Convert the ids back into tokens to SEE the special tokens that got added.
print(bert_tok.convert_ids_to_tokens(encoded["input_ids"]))
# -> ['[CLS]', 'i', 'love', 'token', '##ization', '!', '[SEP]']
# convert_ids_to_tokens maps each integer back to its token string, revealing
# [CLS] at the start and [SEP] at the end -- tokens the model expects.

## 5. Going back: `decode`

After a model produces token IDs (for example, when generating text), you turn them back into a human-readable string with `decode`. This is the reverse of the whole pipeline.

In [ ]:
ids = encoded["input_ids"]

# decode() with special tokens kept in:
print(bert_tok.decode(ids))
# -> [CLS] i love tokenization! [SEP]

# Usually you want the clean text, so skip the special tokens:
print(bert_tok.decode(ids, skip_special_tokens=True))
# -> i love tokenization!

**What this does:** `decode` joins the tokens back into a string (and correctly removes the `##` glue, so `token` + `##ization` becomes `tokenization`). With `skip_special_tokens=True`, the `[CLS]`/`[SEP]` markers are dropped so you get clean, readable output — exactly what you'd show a user after generation.

### ✏️ Exercise

Encode the sentence `"Fine-tuning is fun"` with `bert_tok(...)`, print its `input_ids`, then `decode` them both with and without `skip_special_tokens=True`. Does the round-trip text match the original? (Watch how the hyphen in "Fine-tuning" gets tokenized.)

## 6. A GPT-style tokenizer (`distilgpt2`, byte-level BPE)

Different model families tokenize differently. Let's load `distilgpt2` — a small GPT-2 variant — to see how a **BPE** tokenizer compares to BERT's WordPiece.

Two things to notice:
1. GPT-2 uses a special character (`Ġ`, a visible stand-in for a space) to mark where words begin, instead of BERT's `##` for word-*continuations*.
2. By default GPT-2's tokenizer does **not** add `[CLS]`/`[SEP]` — GPT models work differently (they predict the next token), and use an **end-of-sequence** token instead.

In [ ]:
gpt_tok = AutoTokenizer.from_pretrained("distilgpt2")

print(gpt_tok.tokenize("I love tokenization!"))
# -> ['I', 'Ġlove', 'Ġtoken', 'ization', '!']
# 'Ġ' marks a leading space (start of a new word). 'tokenization' again splits into pieces.

print(gpt_tok("I love tokenization!")["input_ids"])
# -> [40, 1842, 11241, 1634, 0]   (note: NO 101/102 — GPT-2 adds no [CLS]/[SEP] by default)

**What this does:** It loads GPT-2's BPE tokenizer and shows that the *same sentence* produces *different tokens and different ids* than BERT did. The `Ġ` symbol marks the start of a new word (a space). Crucially, the id `40` from GPT-2 and the id `1045` for "i" from BERT are **not interchangeable** — each tokenizer has its own private vocabulary. This is the heart of the "tokenizer must match the model" rule.

In [ ]:
# GPT-2's end-of-sequence (eos) token marks "the text is finished".
print("eos token:", gpt_tok.eos_token)        # -> <|endoftext|>
print("eos token id:", gpt_tok.eos_token_id)  # -> 50256
# During generation, the model emits this token to signal it's done.
# BERT-style models use [SEP]; GPT-style models use eos. Same idea, different name.

## 7. Special tokens, summarized

Special tokens are reserved entries in the vocabulary that carry *structural* meaning rather than ordinary words. The common ones:

| Token | Used by | Purpose |
|-------|---------|---------|
| `[CLS]` | BERT | A "summary" slot at the very start; its final vector is often used for classification. |
| `[SEP]` | BERT | Marks the end of a sentence / separates two sentences. |
| `[PAD]` | BERT | Filler to make sequences in a batch the same length (ignored via the attention mask). |
| `[UNK]` | both | The fallback for something truly unrepresentable (rare with subwords). |
| `<|endoftext|>` (`eos`) | GPT-2 | Marks the end of a sequence; also used as padding for GPT-2. |

You rarely add these by hand — the tokenizer inserts them. But you *do* need to know they exist, because they take up room inside your `max_length` budget.

In [ ]:
# Peek at the special tokens each tokenizer knows about.
print("BERT special tokens:", bert_tok.special_tokens_map)
# -> {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]',
#     'cls_token': '[CLS]', 'mask_token': '[MASK]'}

print("GPT-2 special tokens:", gpt_tok.special_tokens_map)
# -> {'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}
# KEY TAKEAWAY: BERT ships with a real [PAD] token, but GPT-2 has NO pad_token by default.
# Padding a GPT-2 batch without setting one raises an error -- we fix that next section.

## 8. Batching: padding, truncation, `max_length`, and the attention mask

Models train on **batches** — several examples processed together as one rectangular grid of numbers. But sentences have different lengths! To make them rectangular we:

- **Pad** the short ones with a filler token so every row is the same length.
- **Truncate** the long ones so they don't exceed a chosen `max_length`.

The **attention mask** then tells the model "pay attention to the real tokens (`1`), ignore the padding (`0`)." Without it, the model would treat filler as meaningful.

Let's batch two sentences of different lengths with `padding=True`.

In [ ]:
sentences = [
    "I love cats",
    "Tokenization splits words into smaller subword pieces",
]

batch = bert_tok(
    sentences,
    padding=True,        # pad shorter sentences up to the longest one in the batch
    truncation=True,     # cut off anything longer than max_length
    max_length=12,       # the hard cap on sequence length (includes special tokens!)
    return_tensors="pt", # return PyTorch tensors (ready for a model) instead of plain lists
)

print("input_ids shape:", batch["input_ids"].shape)   # -> torch.Size([2, 12])  (2 sentences x 12 tokens)
print(batch["input_ids"])
print(batch["attention_mask"])

**What this does:** With `padding=True`, both sentences are padded to the same length so they form a clean `2 x 12` grid (`return_tensors="pt"` makes them PyTorch tensors). Look at the `attention_mask`: the short sentence's row ends in a run of `0`s — those are the padding positions the model will ignore. The long sentence is truncated to fit `max_length=12`. **`max_length` counts the special tokens too**, so the real content gets a little less room.

In [ ]:
# See the padding tokens explicitly for the FIRST (shorter) sentence.
print(bert_tok.convert_ids_to_tokens(batch["input_ids"][0]))
# -> ['[CLS]', 'i', 'love', 'cats', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
# The trailing [PAD]s line up exactly with the 0s in the attention mask.
# This 1-to-1 correspondence -- real token <-> 1, padding <-> 0 -- is the point of the mask.

In [ ]:
# GPT-2 padding: remember it has NO pad token, so we must set one first.
# A common convention is to reuse the eos token as the pad token.
gpt_tok.pad_token = gpt_tok.eos_token   # fix: give GPT-2 a padding token

gpt_batch = gpt_tok(sentences, padding=True, return_tensors="pt")
print("GPT-2 batch shape:", gpt_batch["input_ids"].shape)   # -> torch.Size([2, 8]) (length varies)
print("attention_mask:\n", gpt_batch["attention_mask"])
# If you FORGET the pad_token line above, GPT-2 raises:
#   "Asking to pad but the tokenizer does not have a padding token." -- now you know the fix.

### ✏️ Exercise

Re-run the BERT batch with `max_length=6` instead of `12`. Which sentence gets truncated, and what tokens survive? Then set `padding="max_length"` (instead of `True`) and observe that *every* sentence is padded to the full `max_length`, even if no sentence is that long. When might each padding strategy be useful for fine-tuning?

## 9. Embeddings: turning IDs into vectors

We now have integer `input_ids`. But an id like `2293` is just a label — the number itself carries no meaning (id 2293 isn't "bigger" or "better" than 2292). The model's **embedding layer** converts each id into a **vector** — a list of floating-point numbers — that *does* carry meaning, learned during training.

An embedding layer is simply a **lookup table**: a big matrix with one row per vocabulary token. To embed a token, you grab its row. Token id `7` → row `7` of the table.

PyTorch gives us this directly as `torch.nn.Embedding(num_tokens, vector_size)`.

In [ ]:
torch.manual_seed(0)   # make the random init reproducible

# A tiny embedding table: a vocabulary of 10 tokens, each mapped to a 4-number vector.
embedding = torch.nn.Embedding(num_embeddings=10, embedding_dim=4)

print("the lookup table (10 rows x 4 columns):")
print(embedding.weight.shape)   # -> torch.Size([10, 4])

# Look up the vectors for token ids 1, 2, and 8.
ids = torch.tensor([1, 2, 8])
vectors = embedding(ids)
print("\nvectors for ids [1, 2, 8]:")
print(vectors)
print("\nshape:", vectors.shape)   # -> torch.Size([3, 4])  (3 tokens, each a 4-d vector)

# A REAL model's embedding layer is the same idea, just bigger:
# BERT-base has vocab 30522 and hidden size 768 -> a table of shape [30522, 768].
# So each of the ~30k tokens becomes a 768-number vector. That's where "meaning" lives.

# The lookup is deterministic: the SAME id always returns the SAME row.
print("same id -> same vector:", torch.equal(embedding(torch.tensor([8])),
                                              embedding(torch.tensor([8]))))  # -> True

**What this does:** `torch.nn.Embedding(10, 4)` creates a 10x4 table of numbers (randomly initialized here; **learned** in a real model). Passing in token ids `[1, 2, 8]` returns rows 1, 2, and 8 — three 4-dimensional vectors. That's *all* an embedding lookup is: indexing into a table. A real model like BERT has a table roughly `30522 x 768` (one 768-number vector per token), and during fine-tuning those numbers keep adjusting to your task. Once you've tokenized text into `input_ids`, the model's very first action is exactly this lookup.

## 10. "Similar meaning ≈ closer vectors" (cosine similarity)

Why bother with vectors instead of plain ids? Because vectors can be **close** or **far** from each other, and a well-trained model arranges them so that **tokens with similar meaning sit near each other**. The usual way to measure closeness is **cosine similarity**: a score from `-1` (opposite) to `1` (identical direction), where higher = more similar.

We'll fake three little vectors by hand to build the intuition (in a real model these would be *learned*).

In [ ]:
import torch.nn.functional as F

# Pretend these are learned embeddings. "cat" and "kitten" point in a similar direction;
# "car" points elsewhere.
cat    = torch.tensor([0.90, 0.10, 0.05])
kitten = torch.tensor([0.85, 0.15, 0.02])
car    = torch.tensor([0.10, 0.05, 0.95])

def cos(a, b):
    # cosine_similarity needs a batch dimension, so we add one with .unsqueeze(0)
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

print("cat vs kitten:", round(cos(cat, kitten), 3))   # -> ~0.99  (very similar)
print("cat vs car:   ", round(cos(cat, car), 3))      # -> ~0.2x  (much less similar)

**What this does:** It computes cosine similarity between three made-up vectors. `cat` and `kitten` score near `1.0` (similar meaning → similar direction), while `cat` and `car` score much lower. This is the geometric intuition behind embeddings: **meaning becomes distance/direction in vector space.** When you fine-tune, you're nudging these vectors (and the layers above them) so this geometry fits *your* task.

### ✏️ Exercise

Make up a fourth vector for `"puppy"` that's close to `kitten` (both are baby animals), and one for `"truck"` that's close to `car`. Compute the four pairwise cosine similarities among `kitten`, `puppy`, `car`, `truck`. Do the "animal" pairs score higher than "animal vs vehicle" pairs? Adjust the numbers until the geometry matches your intuition.

## Common mistakes & how to debug them

- **Tokenizer doesn't match the model.** Loading `bert-base-uncased`'s tokenizer but a GPT-2 model (or vice versa) gives meaningless results — the ids point to the wrong vocabulary. **Fix:** always load both from the *same* checkpoint string, e.g. `AutoTokenizer.from_pretrained(name)` and `AutoModel.from_pretrained(name)` with the same `name`.
- **"Asking to pad but the tokenizer does not have a padding token."** Happens with GPT-2-style tokenizers. **Fix:** `tokenizer.pad_token = tokenizer.eos_token` before batching.
- **Forgetting the attention mask.** If you build batches manually and drop the `attention_mask`, the model treats padding as real text. **Fix:** always pass the `attention_mask` the tokenizer returns straight into the model.
- **`max_length` too small → silent truncation.** Your long examples get cut off and you lose information without an obvious error. **Fix:** check token lengths with `len(tokenizer(text)["input_ids"])` and pick `max_length` accordingly; remember special tokens count toward it.
- **`max_length` too large → out-of-memory / slow.** Compute and memory grow with sequence length. **Fix:** use the smallest `max_length` that keeps most examples intact.
- **Comparing ids across tokenizers.** Id `1045` means different things to different tokenizers. **Fix:** never assume ids are universal; they're private to one tokenizer's vocabulary.

In [ ]:
# Quick debugging recipe: how long (in tokens) are my examples?
examples = ["short one", "a considerably longer example sentence that uses many more tokens than the first"]
for ex in examples:
    n = len(bert_tok(ex)["input_ids"])   # includes [CLS] and [SEP]
    print(f"{n:2d} tokens  <-  {ex!r}")
# Use the distribution of these lengths to pick a sensible max_length for fine-tuning.

**What this does:** It measures the token length of each example (special tokens included). Running this over your whole dataset tells you what `max_length` actually captures most of your data — a simple, practical habit that prevents both silent truncation and wasted memory.

## Summary

- Models only understand **numbers**, so text must become **tokens → token IDs → embedding vectors**.
- **Subword tokenizers** (BPE for GPT-style, WordPiece for BERT-style) split rare/unknown words into known pieces, so nothing is ever truly out-of-vocabulary.
- `AutoTokenizer.from_pretrained(name)` loads the right tokenizer. Key methods:
  - `.tokenize(text)` → human-readable token strings.
  - `tokenizer(text)` → a dict with `input_ids` and `attention_mask` (and auto-added special tokens).
  - `.decode(ids, skip_special_tokens=True)` → back to clean text.
- **Special tokens** (`[CLS]`, `[SEP]`, `[PAD]`, `eos`) carry structure; the tokenizer inserts them and they count toward `max_length`.
- **Padding + truncation + attention mask** turn variable-length text into rectangular batches; the mask marks real (`1`) vs. padding (`0`) tokens. GPT-2 needs `pad_token = eos_token` set manually.
- An **embedding** is a learned **lookup table** (`torch.nn.Embedding`) mapping each token id to a vector; **similar meanings sit closer** (cosine similarity ≈ 1).
- For fine-tuning: **tokenizing your data is the first step**, `max_length` is your memory/speed dial, and a **matching tokenizer + model** avoids a whole class of confusing bugs.

## What to learn next

Now that you can turn text into model-ready numbers, the next notebook puts a whole model behind it: loading models, running them, and reading their outputs.

➡️ **`08_huggingface_transformers_basics.ipynb`** — Hugging Face Transformers basics: `AutoModel`, running a forward pass, and understanding what comes out the other side.